<a href="https://colab.research.google.com/github/LUMII-AILab/NLP_Course/blob/main/notebooks/LLM_as_a_service.ipynb" target="_new"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# Using vLLM end-points

In [ ]:
# The end-point provided by AiLab.lv at IMCS, University of Latvia has an OpenAI-compatible API
!pip install openai
!pip install transformers

In [18]:
from openai import OpenAI
from transformers import GenerationConfig

In [19]:
!hf auth login
# (1) Create a READ token on Hugging Face
# (2) Accept Gemma-3 terms on Hugging Face

User is already logged in. Use `hf auth login --force` to force re-login.


In [20]:
END_POINT = "https://llm.ailab.lv/v1"

client = OpenAI(
    base_url = END_POINT,
    api_key = "YOUR_API_KEY"
)

models = client.models.list()
for model in models.data:
    print(model.id)

    # Read model's original generation_config.json from HF - requires HF login
    config = GenerationConfig.from_pretrained(model.id)
    print("\ttemperature:", config.temperature)
    print("\ttop_p:", config.top_p)
    print("\ttop_k:", config.top_k)

google/gemma-3-27b-it
	temperature: None
	top_p: 0.95
	top_k: 64
TildeAI/TildeOpen-30b
	temperature: None
	top_p: None
	top_k: None
Qwen/Qwen3.8-27B
	temperature: 1.0
	top_p: 0.95
	top_k: 20


In [81]:
CURRENT_IT_MODEL = "google/gemma-3-27b-it"
CURRENT_BASE_MODEL = "TildeAI/TildeOpen-30b"

## Basic inference, metrics, parameters

In [38]:
import time
start = time.perf_counter()

response = client.chat.completions.create(
    model=CURRENT_IT_MODEL,
    messages=[
        {
            "role": "user",
            "content": "Izskaidro trīs īsos punktos, kas ir lielie valodas modeļi."
        }
    ],
    temperature=1.0,
    #extra_body={"chat_template_kwargs": {"enable_thinking": False}}
)

elapsed = time.perf_counter() - start
print(f"Total response time: {elapsed:.2f} sec")

print(CURRENT_IT_MODEL, "\n", response.choices[0].message.content, "\n")

print("Total tokens:", response.usage.total_tokens)
print("\tPrompt tokens:", response.usage.prompt_tokens)
print("\tCompletion tokens:", response.usage.completion_tokens)

if response.usage.completion_tokens_details:
    print("\t\tReasoning tokens:", response.usage.completion_tokens_details.reasoning_tokens)

metrics = response.model_extra.get("metrics")  # vLLM: --enable-per-request-metrics
if metrics:
    print(f"\nTTFT: {metrics.get('time_to_first_token_ms') / 1000:.1f} sec")
    print(f"Tokens/sec: {metrics.get('tokens_per_second'):.1f}")
    print(f"Total time: {metrics.get('generation_time_ms') / 1000:.1f} sec")

Total response time: 12.53 sec
Qwen/Qwen3.8-27B 
 

1. **Tie ir mākslīgi intelekta modeļi, kas apmācīti uz ļoti liela teksta apjoma** — piemēram, grāmatām, rakstiem, tīmekļa lapām un sarunām, lai uzzinātu, kā valoda strādā.

2. **To galvenā darbība ir paredzēt nākamos vārdus vai teksta fragmentus** pēc redzētā konteksta, tāpēc tie var veidot tekstu, atbildēt uz jautājumiem, tulkot un saprast teikumus.

3. **Lielie valodas modeļi ir piemēroti daudzām uzdevumiem** — piemēram, teksta rakstīšanai, kopsavilkumu veidošanai, kodēšanai, datu analīzei un palīdzībai ikdienas darbos. 

Total tokens: 393
	Prompt tokens: 75
	Completion tokens: 318
		Reasoning tokens: 108

TTFT: 0.1 sec
Tokens/sec: 26.3
Total time: 12.0 sec


## Asisstant vs. text completion modes

In [48]:
my_prefix_1 = "The following is a multiple choice question with answers A, B, C, D. The question and answers are in Latvian. Your task: answer the question by replying with 'A', 'B', 'C', or 'D', and nothing else."
my_prompt_1 = "Slavenākais latviešu komponists ir Raimonds \n A: Ešenvalds \n B: Vasks \n C: Pauls \n D: Vītols \n"

# Inference in the asisstant mode
response = client.chat.completions.create(
    model=CURRENT_IT_MODEL,
    messages=[
        {"role": "system", "content": my_prefix_1},
        {"role": "user", "content": my_prompt_1},
    ],
    temperature=0.0,
    max_tokens=1, # Cannot be used in the thinking mode
    extra_body={"chat_template_kwargs": {"enable_thinking": False}}
)

print(CURRENT_IT_MODEL, "\n", response.choices[0].message.content)

# Try Gemma
# Try Qwen with enabled thinking and no max_tokens limit

Qwen/Qwen3.8-27B 
 B


In [56]:
#my_prefix_2 = "Continue the following text. Do not answer as an assistant. Only continue the text directly as a language model, up to 50 words."
my_prompt_2 = "Slavenākais latviešu komponists ir Raimonds"

# Should be done with a base model using client.completions.create instead of client.chat.completions.create
# In case of an IT model, try a prompt-guided chat completion mode (my_prefix_2)
response = client.completions.create(
    model=CURRENT_BASE_MODEL,
    prompt=my_prompt_2,
    temperature=0.75,
    max_tokens=50
)

print(CURRENT_BASE_MODEL, "\n", response.choices[0].text)

TildeAI/TildeOpen-30b 
  Pauls. Viņš ir dzimis, audzis un visu mūžu pavadījis Rīgā. Viņš ir sarakstījis daudz dziesmu. Viņa dziesmas dzied Latvijā un ārzemēs. Tās ir ļoti populāras. Raimondam Paulam patīk arī spēlēt klavieres


In [61]:
my_prompt_3 = "Kurš ir slavenākais latviešu komponists, kura vārds ir Raimonds?"

# Essentially the same but in the asisstant/chat mode
response = client.chat.completions.create(
    model=CURRENT_IT_MODEL,
    messages=[
        {"role": "user", "content": my_prompt_3},
    ],
    temperature=1.0,
    #extra_body={"chat_template_kwargs": {"enable_thinking": False}}
)

print(CURRENT_IT_MODEL, "\n", response.choices[0].message.content)

Qwen/Qwen3.8-27B 
 

Slavenākais latviešu komponists ar vārdu Raimonds ir **Raimonds Pauls** — viens no pazīstamākajiem latviešu mūziķiem un dziesmu autoriem.


# Context-based generation

In [73]:
# User's question
question_1 = "Kāds ir sods par ātruma pārsniegšanu apdzīvotā vietā Latvijā par vairāk nekā 50 km/h? Soda apmēru norādi EUR."

# Assuming that the retrieval part is "some how" already done from Likumi.lv
retrieved_chunks = [
    "Ātruma pārsniegšana apdzīvotā vietā līdz 10 km/st, < 7.5 t - sods ir brīdinājums [Ceļu satiksmes likuma 55.1. pants]",
    "Ātruma pārsniegšana apdzīvotā vietā 11-20 km/st, < 7.5 t - sods ir brīdinājums vai 4 NSV [Ceļu satiksmes likuma 55.3. pants]",
    "Ātruma pārsniegšana apdzīvotā vietā 21-30 km/st, < 7.5 t - sods ir 8 NSV, punktu skaits: 1 [Ceļu satiksmes likuma 55.7. pants]",
    "Ātruma pārsniegšana apdzīvotā vietā 31-40 km/st, < 7.5 t - sods ir 16 NSV, punktu skaits: 2 [Ceļu satiksmes likuma 55.11. pants]",
    "Ātruma pārsniegšana apdzīvotā vietā 41-50 km/st, < 7.5 t - sods ir 32 NSV līdz 44 NSV, punktu skaits: 3 [Ceļu satiksmes likuma 55.15. pants]",
    "Ātruma pārsniegšana apdzīvotā vietā 51-60 km/st, < 7.5 t - sods ir 48 NSV līdz 64 NSV un vadītāja tiesību atņemšana uz 3 mēn., punktu skaits: 4 [Ceļu satiksmes likuma 55.19. pants]",
    "Ātruma pārsniegšana apdzīvotā vietā 61-70 km/st, < 7.5 t - sods ir 144 NSV līdz 192 NSV un vadītāja tiesību atņemšana 9 mēn., punktu skaits: 5 [Ceļu satiksmes likuma 55.23. pants]",
    "Ātruma pārsniegšana apdzīvotā vietā par vairāk nekā 70 km/st - sods ir 280 NSV līdz 400 NSV un vadītāja tiesību atņemšana 12–36 mēn., punktu skaits: 5 [Ceļu satiksmes likuma 55.26.1. pants]",
    "Viena naudas soda vienība (NSV) ir 5 EUR."
]

In [74]:
system_prompt = (
    "Tu esi jautājumu-atbilžu asistents par ceļu satiksmes pārkāpumiem Latvijā. "
    "Atbildi veido, izmantojot doto kontekstu. Atbildi pamato ar attiecīgo likuma pantu, ja tas ir zināms. "
    "Ja atbildi nav iespējams izsecināt no konteksta, tad atbildi, ka tev nav pietiekamas informācijas, lai atbildētu uz šo jautājumu. "
)

context_1 = "\n\n".join(
    f"[{i+1}] {chunk}" for i, chunk in enumerate(retrieved_chunks)
)

user_prompt_1 = f"Konteksts:\n{context_1}\n\nJautājums:\n{question_1}\n\nAtbilde:"

response = client.chat.completions.create(
    model=CURRENT_IT_MODEL,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_1},
    ],
    temperature=0.5
)

answer_1 = response.choices[0].message.content
print(CURRENT_IT_MODEL, "\n", answer_1)

google/gemma-3-27b-it 
 Par ātruma pārsniegšanu apdzīvotā vietā Latvijā par vairāk nekā 50 km/h, ja transportlīdzekļa masa ir mazāka par 7.5 tonnām, sods ir no 280 NSV līdz 400 NSV un vadītāja tiesību atņemšana uz 12–36 mēnešiem (Ceļu satiksmes likuma 55.26.1. pants).

Tā kā viena NSV ir 5 EUR, tad soda apmērs ir no 1400 EUR (280 x 5 EUR) līdz 2000 EUR (400 x 5 EUR).



In [78]:
# Dialog turns
context_2 = user_prompt_1 + answer_1

question_2 = "Vai tad par ātruma pārsniegšanu nedaudz virs 50 km/h, piem., 55km/h uzreiz ir maksimālais sods?"
#question_2 = "Vai ir vēl kādi ātruma pārsniegšanas limiti virs 50 km/h?"
#question_2 = "Kas ir NSV? Izdomā atbildi, pat ja kontekstā nav pietiekamas informācijas."

user_prompt_2 = f"Konteksts:\n{context_2}\n\nJautājums:\n{question_2}\n\nAtbilde:"

response = client.chat.completions.create(
    model=CURRENT_IT_MODEL,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_2},
    ],
    temperature=0.5
)

answer_2 = response.choices[0].message.content
print(CURRENT_IT_MODEL, "\n", answer_2)

Qwen/Qwen3.8-27B 
 

Nē. Ja transportlīdzekļa masa ir mazāka par 7,5 t, tad 55 km/h ātruma pārsniegums apdzīvotā vietā ietilpst **51–60 km/h** pārsnieguma kategorijā, nevis kategorijā “vairāk nekā 70 km/h”.

Saskaņā ar **Ceļu satiksmes likuma 55.19. pantu** par ātruma pārsniegšanu apdzīvotā vietā par **51–60 km/h** transportlīdzeklim ar masu < 7,5 t sods ir:

- **48 NSV līdz 64 NSV**;
- vadītāja tiesību atņemšana uz **3 mēnešiem**;
- punktu skaits: **4**.

Tā kā 1 NSV = 5 EUR, tad naudas sods ir:

- **48 × 5 = 240 EUR**;
- **64 × 5 = 320 EUR**.

Tātad par 55 km/h pārsniegumu sods ir **no 240 EUR līdz 320 EUR** un tiesību atņemšana uz 3 mēnešiem.

Maksimālais no konteksta minētais sods par ātruma pārsniegšanu apdzīvotā vietā ir par **vairāk nekā 70 km/h**: **280–400 NSV**, t.i., **1400–2000 EUR**, un tiesību atņemšana uz **12–36 mēnešiem** saskaņā ar **Ceļu satiksmes likuma 55.26.1. pantu**.


# Multimodal input

In [82]:
from google.colab import files
import base64
import mimetypes

# Upload image
uploaded = files.upload()
image_path = next(iter(uploaded))

# Encode image as base64 (binary bytes to plain text chars) to send it as image_url content
mime_type, _ = mimetypes.guess_type(image_path)
with open(image_path, "rb") as f:
    image_base64 = base64.b64encode(f.read()).decode()

# Multimodal request
# vLLM: --limit-mm-per-prompt
response = client.chat.completions.create(
    model=CURRENT_IT_MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Nolasi, kas rakstīts uz piezīmju lapiņas."
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:{mime_type};base64,{image_base64}"
                    }
                }
            ]
        }
    ]
)

print(CURRENT_IT_MODEL, "\n", response.choices[0].message.content)

Saving 4_Monero.png to 4_Monero.png
google/gemma-3-27b-it 
 Uz piezīmju lapiņas ir rakstīts:

**8F9BBLF5CQNSEVMQGGU**
**NMGYMQGDII28pK9K9TJ**
**Roberto**



# Restricted prediction, optional generation

For MCQ and text classification tasks, make the model predict only selected tokens.

In [102]:
from transformers import AutoTokenizer
import math

prompt = "Slavenākais latviešu komponists ir Raimonds \n\n A: Ešenvalds \n B: Vasks \n C: Pauls \n D: Vītols \n\n Atbilde: "

print(CURRENT_IT_MODEL, "\n")

tokenizer = AutoTokenizer.from_pretrained(CURRENT_IT_MODEL)

labels = ["A", "B", "C", "D"]

# Get token IDs for the labels
# FIXME: verify that each label is exactly one token
label_ids = {
    label: tokenizer.encode(label, add_special_tokens=False)[0]
    for label in labels
}

print(label_ids, "\n")

response = client.completions.create(
    model=CURRENT_IT_MODEL,
    prompt=prompt,
    max_tokens=1,
    temperature=0,
    logprobs=1,
    extra_body={                                       # vLLM-specific params:
        "allowed_token_ids": list(label_ids.values()), # Restrict generation to these tokens
        "logprob_token_ids": list(label_ids.values())  # Return log-probabilities for these tokens
    }
)

logprobs = response.choices[0].logprobs.top_logprobs[0]

# Convert log-probs to probs
probs = {
    label: math.exp(logprobs[label])
    for label in labels
}

for token, prob in probs.items():
    print(f"{token}: {prob:.12f}")

# Re-normalize probs
for label in labels:
    print(f"{label}: {probs[label] / sum(probs.values()):.3f}")

print("\nAnswer:", response.choices[0].text) # Guaranteed to be one of the selected tokens

google/gemma-3-27b-it 

{'A': 236776, 'B': 236799, 'C': 236780, 'D': 236796} 

A: 0.000000000982
B: 0.000000000299
C: 0.000000120815
D: 0.000000000084
A: 0.008
B: 0.002
C: 0.989
D: 0.001

Answer: C
